# Paper 4 — 01 · Build + freeze contrastive sets

Build the five contrastive cells (EXPERIMENT_DESIGN §2): `harm_en`, `benign_en`, `harm_ro`, `benign_ro`, and the EN<->RO `parallel` set. Read sources from Paper 2 (RoSafetyBench) and HarmBench; label behavior with the Paper 2 `gpt-5-mini` judge for the execution probe. Freeze + SHA-256 the sets and the probe split — pre-registration (EXPERIMENT_DESIGN §11).

**Output:** `data/contrastive/<short>/*.jsonl`, `data/splits/probe_split.json`, both SHA-256'd.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


## Sources

- `harm_en`: HarmBench standard + local core from crosslingual `text_en`.
- `benign_en`: XSTest-safe (matches RO over-refusal benign-but-risky semantics).
- `harm_ro` / `benign_ro`: RoSafetyBench (`paper2-benchmark/benchmark/expanded/`).
- `parallel`: RoSafetyBench crosslingual harmful pairs (`category=='harmful'`, patching-only).

In [ ]:
import yaml
cfg = yaml.safe_load((DRIVE_ROOT / 'configs' / 'experiments.yaml').read_text())
cells_cfg = cfg['contrastive_sets']['cells']
cells_cfg

## 1. Build all cells (EN cells pull HarmBench + XSTest from HF)

`with_en=True` builds the EN cells too (needs `datasets` + network — fine on
Colab). RO cells + parallel + harm_en core are deterministic from the committed
Paper 2 files (already frozen locally; SHA-256 in `PREREGISTRATION.md`).

In [ ]:
from contrastive import build_all, make_probe_split
expanded = PAPER2_ROOT / 'benchmark' / 'expanded'
out = CONTRAST_DIR / short
manifest = build_all(expanded, out, with_en=True)
import pprint; pprint.pprint(manifest['cells'])

## 2. Load the anchor (for behavioral generation)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(ANCHOR)
tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(ANCHOR, torch_dtype=torch.bfloat16, device_map='cuda').eval()
print('loaded', ANCHOR)

## 3. Behavioral labels (execution-probe target) — Paper 2 judge

Greedy one completion per prompt, judged refuse/comply by `gpt-5-mini`
(same protocol as Paper 2/3). Idempotent via the judge's on-disk cache.

In [ ]:
from llm_judge import Judge          # Paper 2 src/
from behavioral import behavioral_labels_for_cells, gap_exhibiting_pairs
judge = Judge(model=cfg.get('judge', {}).get('primary', 'openai/gpt-5-mini')
              if isinstance(cfg.get('judge'), dict) else 'openai/gpt-5-mini')
labels_path = behavioral_labels_for_cells(model, tok, judge, out)
print('wrote', labels_path)
print(f'judge calls={judge.total_calls} cache_hits={judge.total_cache_hits}')

## 4. Gap-exhibiting pairs (RO comply + EN refuse) → H1c patching set

In [ ]:
pairs = gap_exhibiting_pairs(labels_path)
print(f'{len(pairs)} / {manifest["cells"]["parallel"]["n"]} parallel pairs exhibit the gap')
if len(pairs) < 15:
    print('WARNING: thin patching set — consider adding the bias subset (EXPERIMENT_LOG 2026-06-01).')

## 5. Freeze probe split + record SHA-256 (append to PREREGISTRATION.md §3)

In [ ]:
split = make_probe_split(out)
print(f"train_en={len(split['train_en'])} eval_en={len(split['eval_en'])} eval_ro={len(split['eval_ro'])}")
print('probe_split sha256:', split['_sha256'])
print('harm_en sha256 :', manifest['cells']['harm_en']['sha256'])
print('benign_en sha256:', manifest['cells']['benign_en']['sha256'])
print('\n>> Append these three SHA-256s to PREREGISTRATION.md section 3 with today\'s date.')